In [ ]:
import requests
import pandas as pd
import glob
import os
import xarray as xr

def epqs_3dep_elev_m(lat, lon):
    """
    USGS EPQS point elevation query (3DEP). Returns elevation in meters.

    Notes:
      - EPQS expects lon/lat as x/y.
      - EPQS elevations are interpolated from 3DEP elevation service.
    """
    url = "https://epqs.nationalmap.gov/v1/json"
    params = {
        "x": float(lon),
        "y": float(lat),
        "units": "Meters",
        "output": "json"
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    j = r.json()

    # EPQS response schema nests things; be defensive:
    # Commonly: j["value"] exists, but some responses use a nested structure.
    if "value" in j:
        return float(j["value"])

    # Fallback for documented structure (varies by EPQS version/output schema):
    try:
        return float(j["USGS_Elevation_Point_Query_Service"]["Elevation_Query"]["Elevation"])
    except Exception as e:
        raise RuntimeError(f"Unexpected EPQS response structure: {j}") from e



In [ ]:

# Example: build a table of PIPS deployments
# Replace these with your actual deployment rows (lat/lon + GPS elevation).
deployments = [
    {"pips": "PIPS2A", "lat": 36.75572945392157, "lon": -98.881402068627472, "gps_elev_m": 482.60570588235294},
    {"pips": "PIPS3A", "lat": 36.37430777372261, "lon": -99.31799872041584, "gps_elev_m": 629.0114797611149},
    {"pips": "PIPS1A", "lat": 36.768747264416305, "lon": -98.881328185654, "gps_elev_m": 497.5151898734177},
    {"pips": "PIPS3B", "lat": 36.366954422583824, "lon": -99.30362241568048, "gps_elev_m": 610.0277218934912}
]

df = pd.DataFrame(deployments)

df["dem_elev_m"] = df.apply(lambda r: epqs_3dep_elev_m(r["lat"], r["lon"]), axis=1)
df["gps_minus_dem_m"] = df["gps_elev_m"] - df["dem_elev_m"]

# Simple flagging threshold (tune as you like)
df["flag"] = df["gps_minus_dem_m"].abs() > 20.0

print(df[["pips", "lat", "lon", "gps_elev_m", "dem_elev_m", "gps_minus_dem_m", "flag"]])

In [ ]:
basedir = '/Users/dawson29/Dropbox/Projects/ICECHIP/obsdata/PIPS_data'
IOP_dirs = glob.glob(basedir + '/IOP*')

In [ ]:
deployments = []
for IOP_dir in IOP_dirs:
    netcdf_dir = os.path.join(IOP_dir, 'netcdf')
    conv_filepaths = glob.glob(netcdf_dir + '/conv*nc')
    print(IOP_dir)
    for conv_filepath in conv_filepaths:
        ds = xr.open_dataset(conv_filepath)
        lat, lon, elev = eval(ds.location)
        deployments.append({"IOP": os.path.basename(IOP_dir), "PIPS": ds.probe_name, "lat": lat, "lon": lon, "gps_elev_m": elev})

df = pd.DataFrame(deployments)

df["dem_elev_m"] = df.apply(lambda r: epqs_3dep_elev_m(r["lat"], r["lon"]), axis=1)
df["gps_minus_dem_m"] = df["gps_elev_m"] - df["dem_elev_m"]

# Simple flagging threshold (tune as you like)
df["flag"] = df["gps_minus_dem_m"].abs() > 20.0

print(df[["IOP", "PIPS", "lat", "lon", "gps_elev_m", "dem_elev_m", "gps_minus_dem_m", "flag"]])
        
    

In [ ]:
pd.set_option('display.max_rows', None)
df

In [ ]:
output_path = os.path.join(basedir, 'gps_vs_3DEP_elevs.csv')
df.to_csv(output_path)
    